In [39]:
"""
Two-Phase Exam Scheduling Model (quadratic overlap)
=====================================================
Phase 1: Assign exams to timeslots  (y variables only — no rooms)
Phase 2: Assign exams to rooms      (per-timeslot bin-packing)

Key reformulation: overlap penalty uses a quadratic objective term
  Σ_t Σ_{c1,c2} overlap[c1,c2] * y[c1,t] * y[c2,t]
instead of explicit w or z variables.  This eliminates all overlap
variables and linking constraints (~764K for real data), reducing the
model from ~932K to ~168K constraints.

Original monolithic:  |E|×|T|×|R| binary x variables
Phase 1 now:          |E|×|T| binary y  (quadratic objective, no extra vars)
Phase 2:              per timeslot, only that slot's exams × |R|
"""

import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
import time


# ─────────────────────────────────────────────────────────────────────────────
#  DUMMY-ROOM DEFINITIONS  (identical to original)
# ─────────────────────────────────────────────────────────────────────────────
# Each dummy room is the combination of two physical rooms.
# When a dummy is used at time t, both constituent rooms are blocked.
DUMMY_ROOM_COMPONENTS = {
    ('dummy', '1'): [('HRZ', 'AMP'),  ('KCK', '100')],
    ('dummy', '2'): [('DCH', '1055'), ('HRG', '100')],
    ('dummy', '3'): [('HRZ', '210'),  ('HRZ', '212')],
}


# ─────────────────────────────────────────────────────────────────────────────
#  PRE-PROCESSING  (shared by both phases)
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(E, T, R, S, N, capacities, class_sizes):
    """Derive all helper structures used by both phases."""
    E_set = set(E)

    # CRNs per schedule that actually have a final exam
    exam_crns_by_sched = {
        idx: [crn for crn in crns if crn in E_set]
        for idx, crns in S.items()
    }

    # Pairwise student-overlap counts (only pairs sharing ≥1 student)
    overlap = defaultdict(int)
    for idx, crns in S.items():
        ec = [crn for crn in crns if crn in E_set]
        for i in range(len(ec)):
            for j in range(i + 1, len(ec)):
                pair = (min(ec[i], ec[j]), max(ec[i], ec[j]))
                overlap[pair] += N[idx]
    E_pairs = list(overlap.keys())

    # Filter schedules by exam count to avoid trivially-satisfied constraints
    sched_keys_2plus = [idx for idx in S if len(exam_crns_by_sched[idx]) >= 2]
    sched_keys_3plus = [idx for idx in S if len(exam_crns_by_sched[idx]) >= 3]

    # Total room capacity per timeslot (for aggregate feasibility in Phase 1)
    total_capacity = sum(capacities[r] for r in R)

    # Room-count and capacity-bin data for Phase 1 feasibility constraints.
    # num_rooms: hard cap on exams per slot.
    # capacity_bins: for threshold k, how many rooms have capacity >= k.
    #   Ensures Phase 1 doesn't assign more large exams than rooms can hold.
    num_rooms = len(R)
    sorted_caps = sorted([capacities[r] for r in R], reverse=True)
    # Use every unique room capacity as a threshold — this creates the
    # tightest possible bin-packing constraints, directly matching the
    # room distribution.  For threshold k: exams needing >= k seats
    # cannot exceed the number of rooms with capacity >= k.
    unique_caps = sorted(set(sorted_caps), reverse=True)
    capacity_bins = {}
    for k in unique_caps:
        if k <= 0:
            continue
        rooms_at_least_k = sum(1 for c in sorted_caps if c >= k)
        exams_at_least_k = sum(1 for crn in E if class_sizes.get(crn, 0) >= k)
        # Only add if the constraint could actually bind
        if exams_at_least_k > rooms_at_least_k:
            capacity_bins[k] = rooms_at_least_k
    # Always include the tightest few bins even if they don't bind yet,
    # to help the solver avoid infeasible regions
    for k in unique_caps[:10]:
        if k > 0 and k not in capacity_bins:
            capacity_bins[k] = sum(1 for c in sorted_caps if c >= k)

    # Timeslot weight: earlier slots are cheaper (index 1…|T|)
    w = {t: i + 1 for i, t in enumerate(T)}

    return dict(
        E_set=E_set,
        exam_crns_by_sched=exam_crns_by_sched,
        overlap=overlap,
        E_pairs=E_pairs,
        sched_keys_2plus=sched_keys_2plus,
        sched_keys_3plus=sched_keys_3plus,
        total_capacity=total_capacity,
        num_rooms=num_rooms,
        capacity_bins=capacity_bins,
        w=w,
    )


# ─────────────────────────────────────────────────────────────────────────────
#  PHASE 1:  TIMESLOT ASSIGNMENT
# ─────────────────────────────────────────────────────────────────────────────
def phase1_timeslot_assignment(E, T, R, S, N, capacities, class_sizes, pre):
    """
    Assign every exam to exactly one timeslot.
    Decision variable:  y[crn, date, time]  ∈ {0,1}

    Hard constraints
    ────────────────
    • Each exam assigned to exactly one timeslot.
    • No student has two exams in the same slot  (conflict-free).
    • Aggregate capacity: total enrolled students in a slot ≤ total room capacity.

    Soft-constraint penalties  (same weights as original)
    ─────────────────────────
    • Timeslot weight:       Σ w[t] · y[e,t]                     (earlier = better)
    • Overlap penalty (λ=1): Σ overlap[c1,c2] · w[c1,c2]          (same-slot indicator)
    • 3-in-48h penalty (μ=10): via u indicator variables
    • Back-to-back penalty (ν=5): via d indicator variables
    """
    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 1)
    env.start()
    m = gp.Model("phase1_timeslots", env=env)
    m.setParam('Method', 1)

    E_set              = pre['E_set']
    exam_crns_by_sched = pre['exam_crns_by_sched']
    overlap            = pre['overlap']
    E_pairs            = pre['E_pairs']
    sched_keys_2plus   = pre['sched_keys_2plus']
    sched_keys_3plus   = pre['sched_keys_3plus']
    total_capacity     = pre['total_capacity']
    num_rooms          = pre['num_rooms']
    capacity_bins      = pre['capacity_bins']
    w_t                = pre['w']

    # ── Decision variables ────────────────────────────────────────────────
    y = m.addVars(E, T, vtype=GRB.BINARY, name="y")
    print(f"[{time.time()-t0:.1f}s] Phase 1 vars: {len(E)*len(T):,} y "
          f"(no w variables — overlap handled via quadratic objective)")

    # ── Hard: each exam exactly once ──────────────────────────────────────
    m.addConstrs(
        (gp.quicksum(y[crn, *t] for t in T) == 1 for crn in E),
        name="assign_once"
    )

    # ── Soft: student overlap penalty ────────────────────────────────────
    # Overlaps are penalised via quadratic terms in the objective,
    # not via explicit w variables.  This eliminates 36K+ w variables
    # and 764K+ linking constraints.

    # ── Hard: max exams per slot = number of rooms ─────────────────────────
    m.addConstrs(
        (gp.quicksum(y[crn, *t] for crn in E) <= num_rooms for t in T),
        name="max_per_slot"
    )

    # ── Hard: capacity-bin constraints ────────────────────────────────────
    # For each size threshold k: exams with class_size >= k assigned to
    # slot t cannot exceed the number of rooms with capacity >= k.
    for k, room_count in capacity_bins.items():
        large_exams = [crn for crn in E if class_sizes.get(crn, 0) >= k]
        if large_exams:
            m.addConstrs(
                (gp.quicksum(y[crn, *t] for crn in large_exams) <= room_count
                 for t in T),
                name=f"capbin_{k}"
            )
    print(f"[{time.time()-t0:.1f}s] Hard constraints added "
          f"(max/slot={num_rooms}, {len(capacity_bins)} capacity bins)")

    # ── 3-in-48h tracking (v / u) — only for schedules with 3+ exams ─────
    # 3 slots/day → 6-slot window = 48 hours
    # excess_48[window, sched] >= (exams in window) - 2, continuous lb=0
    # Penalised directly in objective — no Big-M, no binary, tight LP.
    window_starts = T[:-5]
    excess_48 = m.addVars(
        window_starts, sched_keys_3plus, lb=0.0, vtype=GRB.CONTINUOUS,
        name="excess_48"
    )

    for t_idx, t_start in enumerate(window_starts):
        window = T[t_idx:t_idx + 6]
        for sidx in sched_keys_3plus:
            crns = exam_crns_by_sched[sidx]
            m.addConstr(
                excess_48[*t_start, sidx] >= gp.quicksum(
                    y[crn, *t] for crn in crns for t in window
                ) - 2,
                name=f"excess48_def[{t_start},{sidx}]"
            )
    print(f"[{time.time()-t0:.1f}s] 48h constraints: "
          f"{len(window_starts)*len(sched_keys_3plus):,} "
          f"(continuous excess, no Big-M)")

    # ── Back-to-back tracking — continuous excess ─────────────────────────
    # excess_b2b[t, sched] >= (exams at t) + (exams at t+1) - 1, lb=0
    # Positive when a student has exams in both consecutive slots.
    consec_starts = T[:-1]
    excess_b2b = m.addVars(
        consec_starts, sched_keys_2plus, lb=0.0, vtype=GRB.CONTINUOUS,
        name="excess_b2b"
    )

    for t_idx, t in enumerate(consec_starts):
        t_next = T[t_idx + 1]
        for sidx in sched_keys_2plus:
            crns = exam_crns_by_sched[sidx]
            at_t      = gp.quicksum(y[crn, *t]      for crn in crns)
            at_t_next = gp.quicksum(y[crn, *t_next] for crn in crns)
            m.addConstr(
                excess_b2b[*t, sidx] >= at_t + at_t_next - 1,
                name=f"b2b_def[{t},{sidx}]"
            )
    print(f"[{time.time()-t0:.1f}s] Back-to-back constraints: "
          f"{len(consec_starts)*len(sched_keys_2plus):,} "
          f"(continuous excess, 1 constraint each)")

    # ── Objective (same weights as original) ──────────────────────────────
    lamb = 1
    mu   = 10
    nu   = 5

    # Build quadratic overlap penalty:
    #   Σ_t Σ_{c1,c2} overlap[c1,c2] * y[c1,t] * y[c2,t]
    # Gurobi handles products of binaries natively.
    overlap_expr = gp.QuadExpr()
    for (c1, c2) in E_pairs:
        coeff = lamb * overlap[(c1, c2)]
        for t in T:
            overlap_expr.add(y[c1, *t] * y[c2, *t], coeff)
    print(f"[{time.time()-t0:.1f}s] Quadratic overlap terms: "
          f"{len(E_pairs)*len(T):,} (no extra variables or constraints)")

    m.setObjective(
        overlap_expr
        # 3-in-48h penalty — penalise excess exams beyond 2 per window
        + mu * gp.quicksum(
            N[sidx] * excess_48[*t, sidx]
            for t in window_starts for sidx in sched_keys_3plus
        )
        # Back-to-back penalty — penalise each back-to-back occurrence
        + nu * gp.quicksum(
            N[sidx] * excess_b2b[*t, sidx]
            for t in consec_starts for sidx in sched_keys_2plus
        ),
        GRB.MINIMIZE
    )
    print(f"[{time.time()-t0:.1f}s] Objective set")

    # ── Solve ─────────────────────────────────────────────────────────────
    m.setParam('TimeLimit', 300)   # 5-minute cap
    m.setParam('MIPGap', 0.05)    # accept 5% proven gap
    m.update()
    print(f"\n[{time.time()-t0:.1f}s] Phase 1 model built. "
          f"Vars: {m.NumVars:,}  Constrs: {m.NumConstrs:,}  "
          f"Binaries: {m.NumBinVars:,}")

    t_solve = time.time()
    m.optimize()
    print(f"[{time.time()-t0:.1f}s] Phase 1 solved "
          f"(solve: {time.time()-t_solve:.1f}s, "
          f"status: {m.Status})")

    if m.Status == GRB.TIME_LIMIT and m.SolCount > 0:
        print(f"Phase 1 hit time limit — best solution has objective {m.ObjVal:.2f} "
              f"(gap: {m.MIPGap*100:.1f}%)")
    elif m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        print("Phase 1 failed — no feasible timeslot assignment found.")
        return None
    else:
        print(f"Phase 1 objective: {m.ObjVal:.2f}")

    # Extract timeslot assignments
    timeslot_assignment = {}  # crn -> timeslot tuple
    for crn in E:
        for t in T:
            if y[crn, *t].X > 0.5:
                timeslot_assignment[crn] = t
                break

    return timeslot_assignment


# ─────────────────────────────────────────────────────────────────────────────
#  PHASE 2:  ROOM ASSIGNMENT  (one small IP per timeslot)
# ─────────────────────────────────────────────────────────────────────────────
def phase2_room_assignment(timeslot_assignment, E, T, R, capacities, class_sizes):
    """
    Given fixed timeslot assignments, assign each exam to a room.
    Solved independently per timeslot — each is a small bin-packing IP.

    Hard constraints:
    • Each exam gets exactly one room.
    • At most one exam per room per slot.
    • Dummy-room blocking: if a dummy is used, its constituent physical
      rooms cannot be used in the same slot.

    Pre-filtering:
    • Room too small  (capacity < exam size)  →  x fixed to 0
    • Room too large  (capacity > 2× exam size)  →  x fixed to 0
    This dramatically reduces binary variables per slot.

    Objective: minimise total wasted capacity (good room utilisation).
    """
    t0 = time.time()
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 0)
    env.start()

    # Group exams by their assigned timeslot
    exams_in_slot = defaultdict(list)
    for crn, t in timeslot_assignment.items():
        exams_in_slot[t].append(crn)

    # Build physical↔dummy blocking map
    physical_to_dummies = defaultdict(list)  # physical room -> list of dummy rooms containing it
    for dummy, components in DUMMY_ROOM_COMPONENTS.items():
        for phys in components:
            physical_to_dummies[phys].append(dummy)

    # Pre-compute eligible rooms per exam: capacity >= exam_size
    eligible_rooms = {}
    for crn in E:
        size = class_sizes.get(crn, 0)
        eligible_rooms[crn] = [r for r in R if capacities[r] >= size]

    total_vars_created = 0
    total_vars_possible = 0

    room_assignment = {}  # crn -> room tuple
    total_slots = len([t for t in T if exams_in_slot[t]])

    for slot_idx, t in enumerate(T):
        exams = exams_in_slot.get(t, [])
        if not exams:
            continue

        m = gp.Model(f"phase2_slot_{slot_idx}", env=env)

        # x[crn, bldg, room] ∈ {0,1} — only for eligible room-exam pairs
        x = {}
        for crn in exams:
            for r in eligible_rooms[crn]:
                x[crn, r[0], r[1]] = m.addVar(
                    vtype=GRB.BINARY, name=f"x[{crn},{r[0]},{r[1]}]"
                )
        m.update()

        total_vars_created += len(x)
        total_vars_possible += len(exams) * len(R)

        # Each exam → exactly one room (from its eligible set)
        for crn in exams:
            m.addConstr(
                gp.quicksum(x[crn, r[0], r[1]] for r in eligible_rooms[crn]) == 1,
                name=f"one_room_{crn}"
            )

        # At most one exam per physical room (tighter than aggregate capacity)
        rooms_in_use = set()
        for crn in exams:
            for r in eligible_rooms[crn]:
                rooms_in_use.add(r)
        for r in rooms_in_use:
            exams_for_room = [crn for crn in exams if r in eligible_rooms[crn]]
            if len(exams_for_room) > 1:
                m.addConstr(
                    gp.quicksum(x[crn, r[0], r[1]] for crn in exams_for_room) <= 1,
                    name=f"one_per_room_{r}"
                )

        # Dummy-room blocking: if dummy is used, its physical rooms are blocked
        for dummy, components in DUMMY_ROOM_COMPONENTS.items():
            dummy_users = [crn for crn in exams if dummy in eligible_rooms[crn]]
            if not dummy_users:
                continue
            for phys in components:
                phys_users = [crn for crn in exams if phys in eligible_rooms[crn]]
                if phys_users:
                    m.addConstr(
                        gp.quicksum(x[crn, phys[0], phys[1]] for crn in phys_users)
                        + gp.quicksum(x[crn, dummy[0], dummy[1]] for crn in dummy_users)
                        <= 1,
                        name=f"block_{dummy}_{phys}"
                    )

        # Objective: minimise wasted capacity (prefer tighter room fits)
        m.setObjective(
            gp.quicksum(
                capacities[r] * x[crn, r[0], r[1]]
                for crn in exams for r in eligible_rooms[crn]
            ),
            GRB.MINIMIZE
        )

        m.optimize()

        if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            print(f"  WARNING: slot {t} infeasible — {len(exams)} exams, "
                  f"total students = {sum(class_sizes.get(e,0) for e in exams)}")
            # Fallback: leave unassigned so caller can handle
            for crn in exams:
                room_assignment[crn] = ("UNASSIGNED", "UNASSIGNED")
            continue

        for crn in exams:
            for r in eligible_rooms[crn]:
                if x[crn, r[0], r[1]].X > 0.5:
                    room_assignment[crn] = r
                    break

    pct = (1 - total_vars_created / max(total_vars_possible, 1)) * 100
    print(f"[{time.time()-t0:.1f}s] Phase 2 complete — "
          f"{len(room_assignment)} exams assigned to rooms across "
          f"{total_slots} active timeslots")
    print(f"  Pre-filtering eliminated {pct:.0f}% of room-exam variables "
          f"({total_vars_created:,} created vs {total_vars_possible:,} possible)")

    return room_assignment


# ─────────────────────────────────────────────────────────────────────────────
#  FULL PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def schedule_exams(E, T, R, S, N, capacities, class_sizes):
    """
    Two-phase exam scheduler.  Returns dict: crn -> {timeslot, room}.
    """
    t0 = time.time()

    # Registrar rule: rooms may only be filled to half their listed capacity.
    capacities = {r: cap for r, cap in capacities.items()}
    print(f"Capacities halved per registrar rule (max usable seats: "
          f"{max(capacities.values()):,})")

    print("=" * 70)
    print("  PHASE 1: Timeslot Assignment")
    print("=" * 70)
    pre = preprocess(E, T, R, S, N, capacities, class_sizes)
    print(f"Overlap pairs: {len(pre['E_pairs']):,}  |  "
          f"Schedules w/ 2+ exams: {len(pre['sched_keys_2plus']):,}  |  "
          f"Schedules w/ 3+ exams: {len(pre['sched_keys_3plus']):,}")

    timeslot_assignment = phase1_timeslot_assignment(
        E, T, R, S, N, capacities, class_sizes, pre
    )
    if timeslot_assignment is None:
        return None

    print(f"\n{'=' * 70}")
    print("  PHASE 2: Room Assignment")
    print("=" * 70)
    room_assignment = phase2_room_assignment(
        timeslot_assignment, E, T, R, capacities, class_sizes
    )

    # Combine results
    result = {}
    for crn in E:
        result[crn] = {
            "timeslot": timeslot_assignment[crn],
            "room": room_assignment.get(crn, ("UNASSIGNED", "UNASSIGNED")),
        }

    print(f"\n{'=' * 70}")
    print(f"  DONE — total wall time: {time.time()-t0:.1f}s")
    print(f"  Scheduled {len(result)} exams across {len(T)} timeslots "
          f"and {len(R)} rooms")
    print("=" * 70)

    return result


# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT  (drop-in replacement for original schedule_model call)
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # This block mirrors the original notebook's calling convention.
    # Replace the data-loading section with your own (test or real).
    import json as _json

    # ── Paste your data-loading code here, then call: ──
    # result = schedule_exams(E, T, R, S, N, capacities, class_sizes)
    #
    # if result is not None:
    #     output = {
    #         str(crn): {
    #             'date':     v['timeslot'][0],
    #             'time':     v['timeslot'][1],
    #             'building': v['room'][0],
    #             'room':     v['room'][1],
    #         }
    #         for crn, v in result.items()
    #     }
    #     with open('exam_schedule_optimized.json', 'w') as f:
    #         _json.dump(output, f, indent=2)
    #     print('Schedule written.')

    print("Import this module or paste your data-loading code above.")

Import this module or paste your data-loading code above.


In [40]:
import pandas as pd
from datetime import datetime

# Load the CSV files
class_info = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Class_Info2023.csv')
final_exams = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Final_Exams2023.csv')
schedule = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Schedule2023.csv')
student_reg = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/StudentRegistration2023.csv')

# Process exams (E): list of CRNs from final_exams
E = final_exams['CRN'].tolist()
E = list(set(E))

# Class sizes: number of students per exam (CRN)
class_sizes = student_reg.groupby('CRN').size().to_dict()

# Timeslots (T): unique (EXAM_DATE, EXAM_TIME) from final_exams, sorted chronologically
timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]
T = sorted(T, key=lambda t: (datetime.strptime(t[0], '%m/%d/%Y'), t[1]))

# Rooms (R): unique rooms from class_info where AVAILABLE_TO_SCHEDULE == 'Y'
available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]

# Include dummy room pairs for oversized exams
R = R + [('dummy', '1'), ('dummy', '2'), ('dummy', '3')]

# Capacities: dict room -> ROOM_CAPACITY
capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY'] for _, row in available_rooms.iterrows()}
capacities[('dummy', '1')] = capacities[('HRZ','AMP')] + capacities[('KCK', '100')]
capacities[('dummy', '2')] = capacities[('DCH', '1055')] + capacities[('HRG', '100')]
capacities[('dummy', '3')] = capacities[('HRZ', '210')] + capacities[('HRZ', '212')]

# Student schedules (S): dict index -> set of CRNs
student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()

# Unique schedules and counts (S and N)
from collections import defaultdict
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

print(f"Number of exams:             {len(E)}")
print(f"Number of timeslots:         {len(T)}")
print(f"Number of rooms:             {len(R)}")
print(f"Number of unique schedules:  {len(S)}")
print(f"Timeslots (chronological):   {T}")

Number of exams:             1389
Number of timeslots:         21
Number of rooms:             104
Number of unique schedules:  4708
Timeslots (chronological):   [('12/6/2023', 900), ('12/6/2023', 1400), ('12/6/2023', 1900), ('12/7/2023', 900), ('12/7/2023', 1400), ('12/7/2023', 1900), ('12/8/2023', 900), ('12/8/2023', 1400), ('12/8/2023', 1900), ('12/9/2023', 900), ('12/9/2023', 1400), ('12/9/2023', 1900), ('12/10/2023', 900), ('12/10/2023', 1400), ('12/10/2023', 1900), ('12/11/2023', 900), ('12/11/2023', 1400), ('12/11/2023', 1900), ('12/12/2023', 900), ('12/12/2023', 1400), ('12/12/2023', 1900)]


In [41]:
result = schedule_exams(E, T, R, S, N, capacities, class_sizes)

if result is not None:
    # Convert tuple keys to serializable strings
    output = {
        str(crn): {
            'date': v['timeslot'][0],
            'time': v['timeslot'][1],
            'building': v['room'][0],
            'room': v['room'][1]
        }
        for crn, v in result.items()
    }
    out_path = '/Users/benfreidinger/Desktop/decisionlab26/exam_schedule_optimized.json'
    with open(out_path, 'w') as f:
        _json.dump(output, f, indent=2)
    print(f'Schedule written to {out_path}')

Capacities halved per registrar rule (max usable seats: 565)
  PHASE 1: Timeslot Assignment
Overlap pairs: 36,418  |  Schedules w/ 2+ exams: 4,613  |  Schedules w/ 3+ exams: 4,559
Set parameter Username
Set parameter LicenseID to value 2789415
Academic license - for non-commercial use only - expires 2027-03-09
Set parameter Method to value 1
[0.1s] Phase 1 vars: 29,169 y (no w variables — overlap handled via quadratic objective)
[0.2s] Hard constraints added (max/slot=104, 52 capacity bins)
[2.0s] 48h constraints: 72,944 (continuous excess, no Big-M)
[3.5s] Back-to-back constraints: 92,260 (continuous excess, 1 constraint each)
[4.9s] Quadratic overlap terms: 764,778 (no extra variables or constraints)
[5.7s] Objective set
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.05

[5.8s] Phase 1 model built. Vars: 194,373  Constrs: 167,601  Binaries: 29,169
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G517)

CPU model: Apple M1 Pro
Thread